# 01 - Data Loading & Verification (Dukascopy XAUUSD ticks)

Small, read-only verification that the raw Dukascopy tick export is parsed
correctly and that a single M15 candle can be reconstructed from the
underlying ticks.

This notebook does **not** backtest, does **not** modify source data, and does
**not** save any transformed data to disk. The entire `data/` directory is
treated as READ-ONLY.

## Constants

Only straightforward pandas / NumPy code is used. No prices are modified and
no anomalies are silently fixed.

In [1]:
import numpy as np
import pandas as pd

# --- Constants ---------------------------------------------------------
DATA_PATH = "../data/XAUUSD_DUKAS_2025_01_ticks.csv"
SYMBOL = "XAUUSD"
SOURCE = "Dukascopy"
TIMEFRAME = "15min"
CHECK_CANDLE_UTC = "2025-01-31 21:45:00+00:00"

# Expected headerless source columns, in order.
COLUMNS = ["date", "time", "bid", "ask", "bid_volume", "ask_volume"]

# PASS/FAIL checks, resolved in the final cell.
checks = {}

print("pandas:", pd.__version__)
print("numpy: ", np.__version__)

pandas: 3.0.6
numpy:  2.4.6


## STEP 1 - Load source data (READ-ONLY)

Read the headerless CSV and assign the expected column names in order.
No `mid` price is computed (not needed here) and bid/ask keep their original
floating-point precision.

In [2]:
df = pd.read_csv(DATA_PATH, header=None, names=COLUMNS)

print("Row count:", len(df))
print("Columns:  ", list(df.columns))

print("\nFirst raw row:")
display(df.head(1))

print("Last raw row:")
display(df.tail(1))

checks["data_load"] = list(df.columns) == COLUMNS

Row count: 5798226
Columns:   ['date', 'time', 'bid', 'ask', 'bid_volume', 'ask_volume']

First raw row:


,date,time,bid,ask,bid_volume,ask_volume
0,2025.01.01,23:00:00.455,2625.1,2626.59,0,0


Last raw row:


,date,time,bid,ask,bid_volume,ask_volume
5798225,2025.01.31,21:59:56.320,2798.09,2800.01,0,0


## STEP 2 - Parse UTC timestamps

Build an in-memory `timestamp` column from `date + " " + time` and localize it
explicitly to UTC. The wall-clock value in the file is **not** shifted.
The original `date` and `time` columns are preserved.

In [3]:
df["timestamp"] = pd.to_datetime(
    df["date"] + " " + df["time"],
    format="%Y.%m.%d %H:%M:%S.%f",
).dt.tz_localize("UTC")

print("timestamp dtype:", df["timestamp"].dtype)
print("timezone:       ", df["timestamp"].dt.tz)
print("first timestamp:", df["timestamp"].iloc[0])
print("last  timestamp:", df["timestamp"].iloc[-1])

# Guard against a local -> UTC shift: the first wall clock must be preserved.
expected_first = pd.Timestamp("2025-01-01 23:00:00.455+00:00")
first_preserved = df["timestamp"].iloc[0] == expected_first
print("first wall clock preserved (no shift):", first_preserved)

checks["utc_parse"] = (
    df["timestamp"].dt.tz is not None
    and str(df["timestamp"].dtype).startswith("datetime64")
    and bool(first_preserved)
)

timestamp dtype: datetime64[us, UTC]
timezone:        UTC
first timestamp: 2025-01-01 23:00:00.455000+00:00
last  timestamp: 2025-01-31 21:59:56.320000+00:00
first wall clock preserved (no shift): True


## STEP 3 - Inspect raw data BEFORE sorting

This exposes problems instead of hiding them. Nothing is cleaned, filled,
clipped, interpolated or de-duplicated. Spread is computed in memory only.

In [4]:
raw_is_monotonic = bool(df["timestamp"].is_monotonic_increasing)

total_rows = len(df)
dup_timestamps = int(df["timestamp"].duplicated().sum())
dup_rows = int(df.duplicated(subset=COLUMNS).sum())

bid = df["bid"].to_numpy(dtype="float64", na_value=np.nan)
ask = df["ask"].to_numpy(dtype="float64", na_value=np.nan)

missing_bid = int(df["bid"].isna().sum())
missing_ask = int(df["ask"].isna().sum())
nonfinite_bid = int((~np.isfinite(bid)).sum())
nonfinite_ask = int((~np.isfinite(ask)).sum())
bid_le_zero = int((bid <= 0).sum())
ask_le_zero = int((ask <= 0).sum())
ask_lt_bid = int((ask < bid).sum())

spread = ask - bid  # in memory only

print("total rows:            ", total_rows)
print("first timestamp:       ", df["timestamp"].iloc[0])
print("last  timestamp:       ", df["timestamp"].iloc[-1])
print("already monotonic:     ", raw_is_monotonic)
print("duplicate timestamps:  ", dup_timestamps)
print("exact duplicate rows:  ", dup_rows)
print("missing BID:           ", missing_bid)
print("missing ASK:           ", missing_ask)
print("non-finite BID:        ", nonfinite_bid)
print("non-finite ASK:        ", nonfinite_ask)
print("BID <= 0:              ", bid_le_zero)
print("ASK <= 0:              ", ask_le_zero)
print("ASK < BID:             ", ask_lt_bid)

print("\nspread = ask - bid")
print("min spread:            ", np.nanmin(spread))
print("max spread:            ", np.nanmax(spread))
print("median spread:         ", np.nanmedian(spread))
print("negative spread count: ", int((spread < 0).sum()))
print("zero spread count:     ", int((spread == 0).sum()))

checks["raw_order"] = raw_is_monotonic
checks["bid_ask_validity"] = (
    missing_bid == 0
    and missing_ask == 0
    and nonfinite_bid == 0
    and nonfinite_ask == 0
    and bid_le_zero == 0
    and ask_le_zero == 0
    and ask_lt_bid == 0
)

total rows:             5798226
first timestamp:        2025-01-01 23:00:00.455000+00:00
last  timestamp:        2025-01-31 21:59:56.320000+00:00
already monotonic:      True
duplicate timestamps:   0
exact duplicate rows:   0
missing BID:            0
missing ASK:            0
non-finite BID:         0
non-finite ASK:         0
BID <= 0:               0
ASK <= 0:               0
ASK < BID:              0

spread = ask - bid
min spread:             0.0
max spread:             5.849999999999909
median spread:          0.5
negative spread count:  0
zero spread count:      214


## STEP 4 - Sorted in-memory view

If the source is not already sorted, an in-memory sorted copy is used for the
calculations. Nothing is written back to disk and duplicate timestamps are
**not** removed. The parsed UTC timestamp becomes the `DateTimeIndex`.

In [5]:
if raw_is_monotonic:
    print("Source is already monotonic; no sorting needed.")
else:
    print("Source is NOT monotonic; sorting an in-memory copy only.")

work = df if raw_is_monotonic else df.sort_values("timestamp", kind="stable")
ticks = work.set_index("timestamp")
del work
del df  # free the unindexed frame; 'ticks' is the working view from here on

print("ticks index dtype:", ticks.index.dtype)
print("ticks index tz:   ", ticks.index.tz)
print("ticks shape:      ", ticks.shape)

Source is already monotonic; no sorting needed.
ticks index dtype: datetime64[us, UTC]
ticks index tz:    UTC
ticks shape:       (5798226, 6)


## STEP 5 - Build M15 BID and ASK candles

Resample semantics: frequency = 15min, label = left, closed = left.
So the candle labeled `2025-01-31 21:45:00+00:00` covers exactly
`[2025-01-31 21:45:00+00:00, 2025-01-31 22:00:00+00:00)`.

BID and ASK are resampled separately and never mixed. No mid price is used.
Empty intervals are excluded from DISPLAY only; the source stays untouched.

In [6]:
RESAMPLE_KW = dict(rule="15min", label="left", closed="left")

bid_m15 = ticks["bid"].resample(**RESAMPLE_KW).ohlc()
ask_m15 = ticks["ask"].resample(**RESAMPLE_KW).ohlc()

bid_m15_nonempty = bid_m15.dropna(how="all")
ask_m15_nonempty = ask_m15.dropna(how="all")

print("BID M15 candles (non-empty):", len(bid_m15_nonempty))
print("ASK M15 candles (non-empty):", len(ask_m15_nonempty))

print("\nLast BID candles:")
display(bid_m15_nonempty.tail(3))

BID M15 candles (non-empty): 2014
ASK M15 candles (non-empty): 2014

Last BID candles:


,open,high,low,close
timestamp,,,,
2025-01-31 21:15:00+00:00,2801.52,2802.30,2799.89,2800.86
2025-01-31 21:30:00+00:00,2800.93,2801.18,2798.14,2799.47
2025-01-31 21:45:00+00:00,2799.41,2801.93,2796.61,2798.09


## STEP 6 - `show_candle()`

Reconstruct one M15 candle directly from the raw ticks in
`[candle_start, candle_end)`. Values are always computed from the actual ticks,
never hard-coded. An empty interval is reported clearly.

In [7]:
def candle_bounds(timestamp):
    """Return tz-aware UTC (start, end) for the M15 interval of a candle label."""
    start = pd.Timestamp(timestamp)
    start = start.tz_localize("UTC") if start.tzinfo is None else start.tz_convert("UTC")
    return start, start + pd.Timedelta(minutes=15)


def _ohlc(bid_series, ask_series):
    """Compute OHLC separately for BID and ASK from raw tick series."""
    def side(series):
        return {
            "open": series.iloc[0],
            "high": series.max(),
            "low": series.min(),
            "close": series.iloc[-1],
        }
    return {"BID": side(bid_series), "ASK": side(ask_series)}


def show_candle(timestamp):
    """Print one M15 candle reconstructed from the exact ticks in its interval."""
    start, end = candle_bounds(timestamp)
    window = ticks.loc[(ticks.index >= start) & (ticks.index < end)]

    print("Symbol:      ", SYMBOL)
    print("Source:      ", SOURCE)
    print("Timeframe:   ", TIMEFRAME)
    print("Timezone:    UTC")
    print("Candle start:", start)
    print("Candle end:  ", end)
    print("Tick count:  ", len(window))

    if len(window) == 0:
        print("\nNo ticks in this interval.")
        return None

    print("First tick:  ", window.index[0])
    print("Last tick:   ", window.index[-1])

    values = _ohlc(window["bid"], window["ask"])
    for side_name in ("BID", "ASK"):
        print("\n" + side_name)
        for field in ("open", "high", "low", "close"):
            print(f"  {field.capitalize():<5}:", values[side_name][field])
    return values

## STEP 7 - Independent resample validation

Two independent calculations for `CHECK_CANDLE_UTC`:

- Method A: the pandas M15 resampled BID/ASK DataFrames.
- Method B: raw ticks selected with `timestamp >= start` and
  `timestamp < end`, then first/max/min/last computed directly.

No rounding is applied before comparing.

In [8]:
start, end = candle_bounds(CHECK_CANDLE_UTC)

window = ticks.loc[(ticks.index >= start) & (ticks.index < end)]
b = None  # BID/ASK OHLC computed directly from raw ticks (Method B)

if len(window) == 0 or start not in bid_m15.index:
    print("No ticks available for the check interval; M15 check cannot be evaluated.")
    checks["m15_check"] = False
else:
    a_bid = bid_m15.loc[start]
    a_ask = ask_m15.loc[start]
    b = _ohlc(window["bid"], window["ask"])

    rows = []
    m15_match = True
    for side_name, a in (("BID", a_bid), ("ASK", a_ask)):
        for field in ("open", "high", "low", "close"):
            method_a = a[field]
            method_b = b[side_name][field]
            equal = bool(method_a == method_b)
            m15_match = m15_match and equal
            rows.append(
                {
                    "side": side_name,
                    "field": field,
                    "method_a_resample": method_a,
                    "method_b_raw": method_b,
                    "equal": equal,
                }
            )

    comparison = pd.DataFrame(rows)
    display(comparison)
    print("M15 direct-vs-resample match:", m15_match)

    checks["m15_check"] = bool(m15_match)

,side,field,method_a_resample,method_b_raw,equal
0,BID,open,2799.41,2799.41,True
1,BID,high,2801.93,2801.93,True
2,BID,low,2796.61,2796.61,True
3,BID,close,2798.09,2798.09,True
4,ASK,open,2800.41,2800.41,True
5,ASK,high,2802.64,2802.64,True
6,ASK,low,2799.05,2799.05,True
7,ASK,close,2800.01,2800.01,True


M15 direct-vs-resample match: True


## STEP 8 - Regression reference (BID only)

Values displayed by a previous version of this notebook. They are shown for
comparison **only**: they are never used to build the candle and never treated
as proof of correctness. Any difference is simply reported.

In [9]:
REFERENCE_BID = {"open": 2799.41, "high": 2801.93, "low": 2796.61, "close": 2798.09}

if b is None:
    print("Calculated BID candle unavailable; reference comparison skipped.")
else:
    print("Reference BID OHLC (old notebook):", REFERENCE_BID)
    print("Calculated BID OHLC (this run):   ", b["BID"])

    for field in ("open", "high", "low", "close"):
        diff = b["BID"][field] - REFERENCE_BID[field]
        print(f"  {field:<5}: diff = {diff:+.10f}")

Reference BID OHLC (old notebook): {'open': 2799.41, 'high': 2801.93, 'low': 2796.61, 'close': 2798.09}
Calculated BID OHLC (this run):    {'open': np.float64(2799.41), 'high': np.float64(2801.93), 'low': np.float64(2796.61), 'close': np.float64(2798.09)}
  open : diff = +0.0000000000
  high : diff = +0.0000000000
  low  : diff = +0.0000000000
  close: diff = +0.0000000000


## STEP 9 - Raw tick inspection helper (read-only)

Display raw `timestamp, bid, ask, bid_volume, ask_volume` around a UTC
timestamp. Filters the in-memory frame only; never modifies or saves data.

In [10]:
def show_ticks_around(timestamp, seconds=5):
    """Return raw ticks within `timestamp` +/- `seconds`."""
    center = pd.Timestamp(timestamp)
    center = center.tz_localize("UTC") if center.tzinfo is None else center.tz_convert("UTC")
    lo = center - pd.Timedelta(seconds=seconds)
    hi = center + pd.Timedelta(seconds=seconds)
    window = ticks.loc[(ticks.index >= lo) & (ticks.index <= hi)]
    return window[["bid", "ask", "bid_volume", "ask_volume"]].reset_index()


print(f"Raw ticks around {CHECK_CANDLE_UTC} (+/- 5s):")
display(show_ticks_around(CHECK_CANDLE_UTC, seconds=5).head(20))

Raw ticks around 2025-01-31 21:45:00+00:00 (+/- 5s):


,timestamp,bid,ask,bid_volume,ask_volume
0,2025-01-31 21:44:55.254000+00:00,2799.09,2799.82,0,0
1,2025-01-31 21:44:55.305000+00:00,2799.11,2799.91,0,0
2,2025-01-31 21:44:55.357000+00:00,2799.09,2799.91,0,0
3,2025-01-31 21:44:55.408000+00:00,2799.11,2799.91,0,0
4,2025-01-31 21:44:55.611000+00:00,2799.12,2799.90,0,0
5,2025-01-31 21:44:55.762000+00:00,2799.12,2799.89,0,0
6,2025-01-31 21:44:55.964000+00:00,2799.12,2799.89,0,0
7,2025-01-31 21:44:56.116000+00:00,2799.12,2799.88,0,0
8,2025-01-31 21:44:56.167000+00:00,2799.11,2799.88,0,0
9,2025-01-31 21:44:57.973000+00:00,2799.12,2799.88,0,0


## STEP 10 - Final manual-check output

In [11]:
show_candle(CHECK_CANDLE_UTC)


def pf(value):
    return "PASS" if value else "FAIL"


ready = (
    checks["data_load"]
    and checks["utc_parse"]
    and checks["bid_ask_validity"]
    and checks["m15_check"]
)

print("\n================ VERIFICATION SUMMARY ================")
print("DATA LOAD:                       ", pf(checks["data_load"]))
print("UTC TIMESTAMP PARSING:           ", pf(checks["utc_parse"]))
print("RAW TIMESTAMP ORDER:             ", pf(checks["raw_order"]))
print("BID/ASK VALIDITY:                ", pf(checks["bid_ask_validity"]))
print("M15 DIRECT-VS-RESAMPLE CHECK:    ", pf(checks["m15_check"]))
print("READY FOR MANUAL MT5 COMPARISON: ", "YES" if ready else "NO")
print("======================================================")
print("Note: duplicate timestamps are reported separately and do not fail a check by themselves.")

Symbol:       XAUUSD
Source:       Dukascopy
Timeframe:    15min
Timezone:    UTC
Candle start: 2025-01-31 21:45:00+00:00
Candle end:   2025-01-31 22:00:00+00:00
Tick count:   1039
First tick:   2025-01-31 21:45:00.042000+00:00
Last tick:    2025-01-31 21:59:56.320000+00:00

BID
  Open : 2799.41
  High : 2801.93
  Low  : 2796.61
  Close: 2798.09

ASK
  Open : 2800.41
  High : 2802.64
  Low  : 2799.05
  Close: 2800.01

================ VERIFICATION SUMMARY ================
DATA LOAD:                        PASS
UTC TIMESTAMP PARSING:            PASS
RAW TIMESTAMP ORDER:              PASS
BID/ASK VALIDITY:                 PASS
M15 DIRECT-VS-RESAMPLE CHECK:     PASS
READY FOR MANUAL MT5 COMPARISON:  YES
Note: duplicate timestamps are reported separately and do not fail a check by themselves.
